# General Parser that parse the content, by selecting and overlapping

In [40]:
from pathlib import Path
from pygments.lexers import get_lexer_for_filename
from pygments.util import ClassNotFound
import pandas as pd
from tree_sitter_language_pack import get_parser, get_language

In [41]:
file_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.txt")

In [42]:
def extension_to_language_name(file_path: Path, get_full_name: bool = False) -> str:
    try:
        if not file_path:
            return "unknown"
        lexer = get_lexer_for_filename(file_path)
        if get_full_name:
            return lexer.name
        return lexer.aliases[0] if lexer.aliases else "text"
    except ClassNotFound:
        return file_path.suffix  # Fallback agar extension parse na ho paaye

In [43]:
def general_parser(
    file_path: Path, target: int = 300, overlap: int = 50
) -> list[dict]:
    if target <= overlap:
        raise ValueError("target size must be strictly greater than overlap size")

    with open(file_path, "r", encoding="utf-8") as fr:
        words = fr.read().split()

    if not words:
        return []

    chunks = []
    step = target - overlap

    for i in range(0, len(words), step):
        chunk_words = words[i : i + target]
        content = " ".join(chunk_words)

        metadata = {
            "word_count": len(chunk_words),
            "file_name": file_path.name,
            "file_path": str(file_path),
            "file_type": extension_to_language_name(file_path),
        }
        chunks.append({"content": content, "metadata": metadata})

        # Stop once the end of the word list is reached
        if i + target >= len(words):
            break

    return chunks

In [44]:
# get_file_data(file_path)

In [45]:
file_for = Path("/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.ods")

In [46]:
def parse_csv_and_spreadsheats(file_path: Path):
    # Ensure we get a lowercase extension with the dot stripped or kept depending on your helper function
    language = extension_to_language_name(file_path).lower()

    # Expanded CSV / Tabular formats
    csv_variants = {"csv", ".csv", "tsv", ".tsv"}

    # Expanded Excel formats
    excel_variants = {
        "xls",
        ".xls",
        "xlsx",
        ".xlsx",
        "xlsm",
        ".xlsm",
        "xlsb",
        ".xlsb",
        "ods",
        ".ods",
    }

    data = None
    if language in csv_variants:
        data = pd.read_csv(file_path, nrows=31)
    elif language in excel_variants:
        data = pd.read_excel(file_path, nrows=31)
    else:
        print("not csv or excel:", language)

    if data is not None:
        has_more = len(data) > 30
        data_types = data.dtypes.to_dict()

        metadata = {
            "is_complete": has_more,
            "schema": str(data_types),
            "file_path": str(file_path),
            "file_name": file_path.name,
        }

        result = data.to_dict(orient="records")

        return {"metadata": metadata, "sample-data": result}
    else:
        return None

In [47]:
def parse_ast(file_path: Path):
    file_alias = extension_to_language_name(file_path)
    if not file_alias:
        return None

    lang_name = file_alias
    if not lang_name:
        return None

    try:
        parser = get_parser(lang_name)
        language = get_language(lang_name)
        print(file_alias)
        print(language)


        source_code = file_path.read_text(encoding="utf-8")
        tree = parser.parse(bytes(source_code, "utf-8"))

        return tree
    except Exception:
        print(f"Skipping {file_path.name}: No parser found for language '{lang_name}'")
        return None

In [48]:
def print_ast(node, source_bytes: bytes, indent: str = "", is_last: bool = True):
    if node is None:
        return

    marker = "└── " if is_last else "├── "

    # Extract source code string for leaf nodes (nodes with no children)
    snippet = ""
    if len(node.children) == 0:
        token_text = source_bytes[node.start_byte : node.end_byte].decode(
            "utf-8", errors="replace"
        )
        snippet = f" ➔ {token_text!r}"

    # Format line and column coordinates
    pos = f"[{node.start_point[0] + 1}:{node.start_point[1]}]"

    print(f"{indent}{marker}{node.type} {pos}{snippet}")

    # Prepare indentation string for children
    new_indent = indent + ("    " if is_last else "│   ")

    # Recursively traverse child nodes
    count = len(node.children)
    for i, child in enumerate(node.children):
        print_ast(child, source_bytes, new_indent, is_last=(i == count - 1))

In [49]:
import json
from pathlib import Path
import yaml  # pip install PyYAML
import tomllib
import re

# Use standard library tomllib for Python 3.11+, fallback to tomli for older versions
# try:
# except ImportError:
# import tomli as tomllib  # pip install tomli


def get_project_dependencies(file_path: Path):
    """
    Reads a package manifest file and extracts its dependencies.
    Returns a list of dicts: [{'name': '...', 'version': '...', 'type': '...'}]
    """
    if not file_path.exists():
        return {"error": f"File not found: {file_path}"}

    filename = file_path.name.lower()

    try:
        if filename == "package.json":
            return _parse_package_json(file_path)
        elif filename == "pyproject.toml":
            return _parse_pyproject_toml(file_path)
        elif filename == "pipfile":
            return _parse_pipfile(file_path)
        elif filename.endswith((".yaml", ".yml")):
            return _parse_yaml_env(file_path)
        else:
            return {"error": f"Unsupported file type: {filename}"}
    except Exception as e:
        return {"error": f"Failed to parse {filename}: {str(e)}"}


def _parse_package_json(file_path: Path):
    """Parses JavaScript/TypeScript package.json"""
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    deps = []
    # Standard npm/yarn/pnpm dependency scopes
    scopes = [
        "dependencies",
        "devDependencies",
        "peerDependencies",
        "optionalDependencies",
    ]

    for scope in scopes:
        if scope in data:
            for pkg, version in data[scope].items():
                deps.append({"name": pkg, "version": version, "type": scope})
    return deps

def _parse_deno_json(file_path: Path):
    """Parses Deno deno.json / deno.jsonc"""
    content = file_path.read_text(encoding="utf-8")
    # Strip single-line comments so JSON parsing doesn't fail on .jsonc
    content_clean = re.sub(r"//.*", "", content)

    data = json.loads(content_clean)
    deps = []

    # Deno 2.x standard dependencies block
    if "dependencies" in data:
        for pkg, ver in data["dependencies"].items():
            deps.append({"name": pkg, "version": ver, "type": "dependencies"})

    # Deno import maps / specifiers
    if "imports" in data:
        for alias, target in data["imports"].items():
            deps.append({"name": alias, "version": target, "type": "imports"})

    return deps


def _parse_requirement_string(req_str: str):
    """Splits 'ipykernel>=7.3.0' into name: 'ipykernel', version: '>=7.3.0'"""
    # Regex splits at the first operator (<, >, =, ~, !, ^)
    match = re.match(r"^([a-zA-Z0-9_\-\.\[\]]+)\s*(.*)$", req_str.strip())
    if match:
        pkg_name, version_spec = match.groups()
        return pkg_name, version_spec if version_spec else "*"
    return req_str, "*"


def _parse_pyproject_toml(file_path: Path):
    """Parses Python pyproject.toml (PEP 621, PEP 735, and Poetry)"""
    with open(file_path, "rb") as f:
        data = tomllib.load(f)

    deps = []

    # 1. Standard PEP 621 dependencies
    if "project" in data:
        project = data["project"]
        if "dependencies" in project:
            for item in project["dependencies"]:
                pkg_name, ver = _parse_requirement_string(item)
                deps.append({"name": pkg_name, "version": ver, "type": "dependencies"})

        if "optional-dependencies" in project:
            for group, pkgs in project["optional-dependencies"].items():
                for item in pkgs:
                    pkg_name, ver = _parse_requirement_string(item)
                    deps.append(
                        {
                            "name": pkg_name,
                            "version": ver,
                            "type": f"optional ({group})",
                        }
                    )

    # 2. PEP 735 Dependency Groups (uv, hatch, etc.)
    if "dependency-groups" in data:
        for group_name, pkgs in data["dependency-groups"].items():
            for item in pkgs:
                if isinstance(item, str):
                    pkg_name, ver = _parse_requirement_string(item)
                    deps.append(
                        {
                            "name": pkg_name,
                            "version": ver,
                            "type": f"group ({group_name})",
                        }
                    )

    # 3. Poetry dependencies
    if "tool" in data and "poetry" in data["tool"]:
        poetry = data["tool"]["poetry"]
        if "dependencies" in poetry:
            for pkg, version in poetry["dependencies"].items():
                if pkg.lower() != "python":
                    ver_str = (
                        version["version"] if isinstance(version, dict) else version
                    )
                    deps.append(
                        {
                            "name": pkg,
                            "version": ver_str,
                            "type": "dependencies",
                        }
                    )

        if "group" in poetry:
            for group_name, group_data in poetry["group"].items():
                if "dependencies" in group_data:
                    for pkg, version in group_data["dependencies"].items():
                        ver_str = (
                            version["version"] if isinstance(version, dict) else version
                        )
                        deps.append(
                            {
                                "name": pkg,
                                "version": ver_str,
                                "type": f"dev-group ({group_name})",
                            }
                        )

    return deps


def _parse_pipfile(file_path: Path):
    """Parses Python Pipfile"""
    with open(file_path, "rb") as f:
        data = tomllib.load(f)

    deps = []
    scopes = ["packages", "dev-packages"]

    for scope in scopes:
        if scope in data:
            for pkg, version in data[scope].items():
                # version can be a dict if extras are specified, e.g., {'version': '*', 'extras': ['dev']}
                version_str = (
                    version.get("version", "*")
                    if isinstance(version, dict)
                    else version
                )
                deps.append({"name": pkg, "version": version_str, "type": scope})
    return deps


def _parse_yaml_env(file_path: Path):
    """Parses Conda environment.yml"""
    with open(file_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    deps = []
    if data and "dependencies" in data:
        for item in data["dependencies"]:
            if isinstance(item, str):
                # Usually looks like "numpy=1.21.0" or just "numpy"
                parts = item.replace(">=", "=").replace("<=", "=").split("=")
                pkg = parts[0].strip()
                version = parts[1].strip() if len(parts) > 1 else "*"
                deps.append({"name": pkg, "version": version, "type": "dependencies"})
            elif isinstance(item, dict) and "pip" in item:
                # Handle pip nested dependencies inside conda yaml
                for pip_req in item["pip"]:
                    parts = pip_req.replace(">=", "==").split("==")
                    pkg = parts[0].strip()
                    version = parts[1].strip() if len(parts) > 1 else "*"
                    deps.append(
                        {"name": pkg, "version": version, "type": "pip-dependencies"}
                    )

    return deps

In [50]:
get_project_dependencies(Path("/home/user/Documents/Project_5/backend/experiments/data/code/conda.yml"))

[{'name': 'python', 'version': '3.12', 'type': 'dependencies'},
 {'name': 'numpy', 'version': '*', 'type': 'dependencies'},
 {'name': 'pandas', 'version': '*', 'type': 'dependencies'},
 {'name': 'requests', 'version': '*', 'type': 'dependencies'},
 {'name': 'pip', 'version': '*', 'type': 'dependencies'},
 {'name': 'fastapi', 'version': '*', 'type': 'pip-dependencies'},
 {'name': 'uvicorn', 'version': '*', 'type': 'pip-dependencies'}]

In [ ]:
import json
import re
from pathlib import Path
import xml.etree.ElementTree as ET

# TOML parsing (built-in for Python 3.11+, fallback to tomli)
import tomllib
# try:
# except ImportError:
#     import tomli as tomllib

# Optional PyYAML for Conda environment files
try:
    import yaml
except ImportError:
    yaml = None


def get_project_dependencies(file_path: Path):
    """
    Auto-detects project type and extracts dependencies from Python, JS/TS,
    Deno, Go, and Java/JVM manifest files.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        return {"error": f"File not found: {file_path}"}

    filename = file_path.name.lower()

    try:
        # --- Go ---
        if filename == "go.mod":
            return _parse_go_mod(file_path)

        # --- Python ---
        elif filename == "pyproject.toml":
            return _parse_pyproject_toml(file_path)
        elif filename == "pipfile":
            return _parse_pipfile(file_path)
        elif filename in ("environment.yml", "environment.yaml"):
            return _parse_yaml_env(file_path)
        elif filename == "requirements.txt" or (filename.startswith("requirements") and filename.endswith(".txt")):
            return _parse_requirements_txt(file_path)

        # --- JavaScript / TypeScript / Deno ---
        elif filename == "package.json":
            return _parse_package_json(file_path)
        elif filename in ("deno.json", "deno.jsonc"):
            return _parse_deno_json(file_path)

        # --- Java / JVM Ecosystem ---
        elif filename == "pom.xml":
            return _parse_maven_pom(file_path)
        elif filename in ("build.gradle", "build.gradle.kts"):
            return _parse_gradle_build(file_path)
        elif filename == "libs.versions.toml":
            return _parse_gradle_catalog(file_path)
        elif filename == "build.sbt":
            return _parse_sbt_build(file_path)

        else:
            return {"error": f"Unsupported or unrecognized file type: {file_path.name}"}

    except Exception as e:
        return {"error": f"Failed to parse {file_path.name}: {str(e)}"}


# =====================================================================
# Go Handlers
# =====================================================================

def _parse_go_mod(file_path: Path):
    """Parses Go module definitions (go.mod) including direct and indirect dependencies"""
    content = file_path.read_text(encoding="utf-8")
    deps = []
    
    in_require_block = False

    for line in content.splitlines():
        line = line.strip()
        
        # Skip comments or empty lines
        if not line or line.startswith("//"):
            continue

        # Handle require block brackets
        if line.startswith("require ("):
            in_require_block = True
            continue
        elif in_require_block and line == ")":
            in_require_block = False
            continue

        # Extract requirements
        if in_require_block:
            _extract_go_dep(line, deps)
        elif line.startswith("require "):
            req_expr = line[len("require "):].strip()
            _extract_go_dep(req_expr, deps)

    return deps


def _extract_go_dep(line: str, deps: list):
    """Helper to extract package name, version, and direct/indirect state from a go.mod line"""
    is_indirect = "// indirect" in line
    clean_line = line.split("//")[0].strip()
    parts = clean_line.split()

    if len(parts) >= 2:
        pkg, version = parts[0], parts[1]
        deps.append({
            "name": pkg,
            "version": version,
            "type": "indirect" if is_indirect else "direct"
        })


# =====================================================================
# Python Handlers
# =====================================================================

def _parse_requirement_string(req_str: str):
    """Splits 'ipykernel>=7.3.0' into name: 'ipykernel', version: '>=7.3.0'"""
    match = re.match(r"^([a-zA-Z0-9_\-\.\[\]]+)\s*(.*)$", req_str.strip())
    if match:
        pkg, ver = match.groups()
        return pkg, ver if ver else "*"
    return req_str, "*"


def _parse_requirements_txt(file_path: Path):
    """Parses standard requirements.txt files and its variants"""
    content = file_path.read_text(encoding="utf-8")
    deps = []

    for line in content.splitlines():
        # Strip comments
        line = line.split("#")[0].strip()

        # Skip empty lines, sub-requirements flags (-r), editables (-e), or indexes (-i, --extra-index-url)
        if not line or line.startswith("-") or line.startswith("--"):
            continue

        # Strip environment markers (e.g., ; python_version > '3.8')
        if ";" in line:
            line = line.split(";")[0].strip()

        if line:
            pkg, ver = _parse_requirement_string(line)
            deps.append({"name": pkg, "version": ver, "type": "dependencies"})

    return deps


def _parse_pyproject_toml(file_path: Path):
    with open(file_path, "rb") as f:
        data = tomllib.load(f)

    deps = []

    if "project" in data:
        project = data["project"]
        for item in project.get("dependencies", []):
            pkg, ver = _parse_requirement_string(item)
            deps.append({"name": pkg, "version": ver, "type": "dependencies"})

        for group, pkgs in project.get("optional-dependencies", {}).items():
            for item in pkgs:
                pkg, ver = _parse_requirement_string(item)
                deps.append({"name": pkg, "version": ver, "type": f"optional ({group})"})

    if "dependency-groups" in data:
        for group, pkgs in data["dependency-groups"].items():
            for item in pkgs:
                if isinstance(item, str):
                    pkg, ver = _parse_requirement_string(item)
                    deps.append({"name": pkg, "version": ver, "type": f"group ({group})"})

    if "tool" in data and "poetry" in data["tool"]:
        poetry = data["tool"]["poetry"]
        for pkg, ver in poetry.get("dependencies", {}).items():
            if pkg.lower() != "python":
                ver_str = ver["version"] if isinstance(ver, dict) else ver
                deps.append({"name": pkg, "version": ver_str, "type": "dependencies"})

        for group_name, group_data in poetry.get("group", {}).items():
            for pkg, ver in group_data.get("dependencies", {}).items():
                ver_str = ver["version"] if isinstance(ver, dict) else ver
                deps.append({"name": pkg, "version": ver_str, "type": f"dev-group ({group_name})"})

    return deps


def _parse_pipfile(file_path: Path):
    with open(file_path, "rb") as f:
        data = tomllib.load(f)

    deps = []
    for scope in ("packages", "dev-packages"):
        for pkg, ver in data.get(scope, {}).items():
            ver_str = ver.get("version", "*") if isinstance(ver, dict) else ver
            deps.append({"name": pkg, "version": ver_str, "type": scope})
    return deps


def _parse_yaml_env(file_path: Path):
    if not yaml:
        raise ImportError("PyYAML is required to parse environment.yml (pip install PyYAML)")

    with open(file_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    deps = []
    for item in data.get("dependencies", []):
        if isinstance(item, str):
            parts = item.replace(">=", "=").replace("<=", "=").split("=")
            deps.append({"name": parts[0].strip(), "version": parts[1].strip() if len(parts) > 1 else "*", "type": "dependencies"})
        elif isinstance(item, dict) and "pip" in item:
            for pip_req in item["pip"]:
                parts = pip_req.replace(">=", "==").split("==")
                deps.append({"name": parts[0].strip(), "version": parts[1].strip() if len(parts) > 1 else "*", "type": "pip-dependencies"})
    return deps


# =====================================================================
# JS / TS / Deno Handlers
# =====================================================================

def _parse_package_json(file_path: Path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    deps = []
    scopes = ["dependencies", "devDependencies", "peerDependencies", "optionalDependencies"]
    for scope in scopes:
        for pkg, ver in data.get(scope, {}).items():
            deps.append({"name": pkg, "version": ver, "type": scope})
    return deps


def _parse_deno_json(file_path: Path):
    content = file_path.read_text(encoding="utf-8")
    content_clean = re.sub(r"//.*", "", content)
    data = json.loads(content_clean)

    deps = []
    for pkg, ver in data.get("dependencies", {}).items():
        deps.append({"name": pkg, "version": ver, "type": "dependencies"})
    for alias, target in data.get("imports", {}).items():
        deps.append({"name": alias, "version": target, "type": "imports"})
    return deps


# =====================================================================
# Java / JVM Handlers
# =====================================================================

def _parse_maven_pom(file_path: Path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    for elem in root.iter():
        if "}" in elem.tag:
            elem.tag = elem.tag.split("}", 1)[1]

    properties = {}
    props_node = root.find("properties")
    if props_node is not None:
        for child in props_node:
            properties[child.tag] = (child.text or "").strip()

    deps = []
    deps_node = root.find("dependencies")
    if deps_node is not None:
        for dep in deps_node.findall("dependency"):
            group_id = dep.findtext("groupId", "").strip()
            artifact_id = dep.findtext("artifactId", "").strip()
            version = dep.findtext("version", "*").strip()
            scope = dep.findtext("scope", "compile").strip()

            if version.startswith("${") and version.endswith("}"):
                version = properties.get(version[2:-1], version)

            deps.append({"name": f"{group_id}:{artifact_id}", "version": version, "type": scope})

    return deps


def _parse_gradle_build(file_path: Path):
    content = file_path.read_text(encoding="utf-8")
    deps = []

    pattern = r'(\w+)\s*\(?\s*[\'"]([^\'"\s:]+):([^\'"\s:]+):([^\'"\s:]+)[\'"]\s*\)?'
    for match in re.finditer(pattern, content):
        scope, group, artifact, ver = match.groups()
        deps.append({"name": f"{group}:{artifact}", "version": ver, "type": scope})

    return deps


def _parse_gradle_catalog(file_path: Path):
    with open(file_path, "rb") as f:
        data = tomllib.load(f)

    versions = data.get("versions", {})
    deps = []

    for alias, info in data.get("libraries", {}).items():
        if isinstance(info, str):
            parts = info.split(":")
            deps.append({"name": f"{parts[0]}:{parts[1]}" if len(parts) >= 2 else info, "version": parts[2] if len(parts) >= 3 else "*", "type": "catalog"})
        elif isinstance(info, dict):
            module = info.get("module", f"{info.get('group', '')}:{info.get('name', '')}")
            ver = info.get("version", "*")
            if isinstance(ver, dict) and "ref" in ver:
                ver = versions.get(ver["ref"], "*")
            deps.append({"name": module, "version": str(ver), "type": "catalog"})

    return deps


def _parse_sbt_build(file_path: Path):
    content = file_path.read_text(encoding="utf-8")

    # 1. Remove comments to avoid parsing commented-out code
    clean_content = re.sub(r"/\*.*?\*/", "", content, flags=re.DOTALL)
    clean_content = re.sub(r"//.*", "", clean_content)

    # 2. Extract variable definitions inside ext { ... }
    ext_vars = {}
    ext_match = re.search(r"ext\s*\{([^}]+)\}", clean_content, re.DOTALL)
    if ext_match:
        ext_body = ext_match.group(1)
        for match in re.finditer(r"([a-zA-Z0-9_]+)\s*=\s*['\"]([^'\"]+)['\"]", ext_body):
            ext_vars[match.group(1)] = match.group(2)

    # 3. Extract dependencies across multi-line function calls
    dep_pattern = r"(\w+)\s*\(\s*['\"]([^'\"]+)['\"]"
    deps = []

    for match in re.finditer(dep_pattern, clean_content, re.DOTALL):
        config_type, raw_dep = match.groups()
        
        # Remove line breaks and spaces inside the string literal
        raw_dep = "".join(raw_dep.split())
        parts = raw_dep.split(":")
        
        if len(parts) < 2:
            continue

        group, artifact = parts[0], parts[1]
        raw_version = parts[2] if len(parts) >= 3 else "*"

        # 4. Substitute ${variable} placeholders with their ext values
        resolved_version = raw_version
        for var_name, var_value in ext_vars.items():
            resolved_version = resolved_version.replace(f"${{{var_name}}}", var_value)
            resolved_version = resolved_version.replace(f"${var_name}", var_value)

        deps.append({
            "name": f"{group}:{artifact}",
            "version": resolved_version,
            "type": config_type
        })

    return deps

In [55]:
get_project_dependencies(Path("/home/user/Documents/Project_5/backend/experiments/data/code/build.gradle"))

[{'name': 'org.springframework.boot:spring-boot-dependencies',
  'version': '${springBootVersion}',
  'type': 'mavenBom'},
 {'name': 'org.testcontainers:testcontainers-bom',
  'version': '${testcontainersVersion}',
  'type': 'mavenBom'},
 {'name': 'org.postgresql:postgresql',
  'version': '${postgresqlVersion}',
  'type': 'runtimeOnly'},
 {'name': 'org.springdoc:springdoc-openapi-starter-webmvc-ui',
  'version': '${springdocVersion}',
  'type': 'implementation'},
 {'name': 'org.mapstruct:mapstruct',
  'version': '${mapstructVersion}',
  'type': 'implementation'},
 {'name': 'org.mapstruct:mapstruct-processor',
  'version': '${mapstructVersion}',
  'type': 'annotationProcessor'},
 {'name': 'org.yaml:snakeyaml', 'version': '2.4', 'type': 'implementation'},
 {'name': 'org.postgresql:postgresql',
  'version': '${postgresqlVersion}',
  'type': 'runtimeOnly'}]